In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from natsort import natsorted
import numpy as np
import matplotlib.animation as animation
import xarray as xr
import h5py
import imageio
import matplotlib
import gc
import sys
import io
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from scipy.optimize import curve_fit
import scipy.integrate
import re
import scipy.ndimage


from AutomationModule import * 

import AutomationModule as AM
plt.rcParams.update({'font.size': 22})  

tidy3dAPI = os.environ["API_TIDY3D_KEY"]
plt.rc('font', family='Arial')


In [2]:
folder_path = r"/mnt/fast/Codes/tidy3d/data/20261124_LSU_T_L_Tests_n_2.90_ff_0.2248_periodic_25a"

In [3]:
n_values = []
ff_values = []
size_values = []
values = {}
reference_object = AM.loadFromFile(key = tidy3dAPI, file_path=os.path.join(folder_path, "LSU_reference.txt"),get_ref=False)
reference_entry = reference_object.sim_data['flux2'].flux
reference_exit = reference_object.sim_data['flux1'].flux
# Loop through all files in the folder
for dirpath, dirnames, filenames in os.walk(folder_path):
      for filename in filenames:
            
        try:
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff = float(re.search(r'ff_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            size = float(re.search(r'size_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff_values.append(ff)
            n_values.append(n_value)
            size_values.append(size)
            #Retrieve simulation data 
            if os.path.isfile(os.path.join(dirpath, filename)):
              file=os.path.join(dirpath, filename)
              structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
              sim_data_i = structure_1.sim_data
              transmission_entry = sim_data_i['flux2'].flux
              transmission_exit = sim_data_i['flux1'].flux
              if n_value not in values.keys():
                  values[n_value] = {}
              if size not in values[n_value].keys():
                  values[n_value][size] = {}
              if ff not in values[n_value][size].keys():
                   values[n_value][size][ff] = {}

              values[n_value][size][ff]['entry'] = transmission_entry.values/reference_entry.values
              values[n_value][size][ff]['exit'] = transmission_exit.values/reference_exit.values
        except Exception as e:
            print("Error:", e)
            continue
       

# After the loop, get unique values as arrays
n_unique = np.unique(n_values)
ff_unique = np.unique(ff_values)
size_unique = np.unique(size_values)




Configuration saved successfully.


16:00:09 CET Billed flex credit cost: 0.446.

             Note: the task cost pro-rated due to early shutoff was below the   
             minimum threshold, due to fast shutoff. Decreasing the simulation  
             'run_time' should decrease the estimated, and correspondingly the  
             billed cost of such tasks.

Error: 'NoneType' object has no attribute 'group'
Configuration saved successfully.


16:00:12 CET Billed flex credit cost: 44.273.

             Note: the task cost pro-rated due to early shutoff was below the   
             minimum threshold, due to fast shutoff. Decreasing the simulation  
             'run_time' should decrease the estimated, and correspondingly the  
             billed cost of such tasks.

... (59 outputs left out)

16:04:53 CET Loading results from                                               
             ../../../output/data/20261124_LSU_T_L_Tests_n_2.90_ff_0.2248_period
             ic_25a/n_2.90/LSU_cyl_n_2.90_ff_0.2248_size_1.txt/Data.hdf5

16:05:20 CET WARNING: Simulation final field decay value of 4.74e-06 is greater 
             than the simulation shutoff threshold of 1e-09. Consider running   
             the simulation again with a larger 'run_time' duration for more    
             accurate results.                                                  

16:05:27 CET Billed flex credit cost: 63.772.

             Note: the task cost pro-rated due to early shutoff was below the   
             minimum threshold, due to fast shutoff. Decreasing the simulation  
             'run_time' should decrease the estimated, and correspondingly the  
             billed cost of such tasks.

In [4]:
print("Unique n values:", n_unique)
print("Unique ff values:", ff_unique)
print("Unique size values:", size_unique)

Unique n values: [2.9]
Unique ff values: [0.2248]
Unique size values: [0.02 0.05 0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65
 0.7  0.75 0.8  0.85 0.9  0.95 1.  ]


In [6]:
data = {
        'transmission_data':values,
        "raw_freqs":sim_data_i.simulation.monitors[0].freqs,
        "sizes":np.array(size_unique),
        "ff":np.array(ff_unique),
        "cell_size":53.10,
        "n_values":np.array(n_unique)
    }

os.makedirs("./data",exist_ok=True)

create_hdf5_from_dict(data,rf"./data/20261125_ff_{ff_unique[0]:.4f}_circular_rods_n_{n_unique[0]:.2f}.h5")
